# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shaheerkhan1117/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why


I'm choosing **Lane 2: Refresh / Content Opportunity Scoring** — which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring?

I'm picking this lane because it's a direct extension of the pipeline I already ran in notebooks 01 and 02: that pipeline builds a ranked refresh queue and compares a hand-written rule baseline against a learned model. I already have a first, working version of exactly this problem, real numbers behind it, and a client-holdout validation split in place — so I can spend the next 7 weeks deepening it (better features, the full warehouse, leakage checks, a real action queue) rather than starting a new problem from scratch.

In [1]:
import os, sys, subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/shaheerkhan1117/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Starter dataset: {df.shape[0]:,} rows, {df.shape[1]} columns, {df['client_id'].nunique()} clients")



Starter dataset: 30,000 rows, 44 columns, 32 clients


## 2. The question: decision, action, cost of a wrong call

**Decision:** out of a client's full page inventory, which pages should a content editor
prioritize for refresh this week?

**Who acts, and how:** a content editor or SEO strategist works from a ranked queue instead of
scanning the whole site — they pick the top N pages and take an action per page (rewrite,
expand, prune, or just monitor).

**Cost of a wrong call:**
- False positive (flagged, but it wasn't worth fixing): wastes editor hours, and the queue loses
  credibility if it happens often.
- False negative (a genuinely declining, high-value page never surfaces): the page keeps losing
  visibility silently, costing weeks of clicks that were cheap to fix early.

Editor time is the scarce resource, so noisy false positives matter as much as missed declines —
this is a triage/ranking problem, not a "catch everything" problem.

**Why data or ML helps at all:** no editor can eyeball every page — see the pool size below. A
hand-written rule is transparent but crude: my own baseline run (notebook 01) got it right on
only 24% of its top-50 picks. The real pattern isn't one clean threshold, it's many weak signals
(freshness, position, impressions, CTR) interacting — exactly where ML earns its place over an
if-statement.

In [2]:
visible = df[df["impressions_90d"] >= 100]
print(f"Visible pages (impressions_90d >= 100): {len(visible):,} of {len(df):,} "
      f"({len(visible)/len(df):.1%})")
print(f"At a generous 50 pages reviewed per editor per week, this pool alone would take "
      f"~{len(visible)/50:.0f} editor-weeks to review by hand once.")



Visible pages (impressions_90d >= 100): 22,006 of 30,000 (73.4%)
At a generous 50 pages reviewed per editor per week, this pool alone would take ~440 editor-weeks to review by hand once.


## 3. Quick look at the data (2-3 real numbers)

Three numbers from the starter dataset that make this lane worth the next 7 weeks:

In [5]:
import sys, subprocess, json

# Install reportlab if not already installed
try:
    import reportlab
except ImportError:
    print("Installing reportlab...")
    subprocess.run([sys.executable, "-m", "pip", "install", "reportlab"], check=True)
    print("reportlab installed.")

# 1) Scale of the problem
declining_rate = (df["trend_direction"] == "down").mean()
print(f"1) Declining rate across all {len(df):,} pages: {declining_rate:.1%}")

# 2) Addressable pool (from section 2 above)
print(f"2) Visible/addressable pool: {len(visible):,} pages ({len(visible)/len(df):.1%} of the dataset)")

# 3) Does a learned model actually beat the hand-written rule? (re-run my own pipeline)
process = subprocess.run([sys.executable, "scripts/run_all.py"], capture_output=True, text=True)
if process.returncode != 0:
    print("Error running scripts/run_all.py:")
    print("STDOUT:", process.stdout)
    print("STDERR:", process.stderr)
    raise subprocess.CalledProcessError(process.returncode, process.args, output=process.stdout, stderr=process.stderr)

res = json.load(open("outputs/model_results.json"))
base = res["baseline"]["baseline_precision_at_50"]
rf   = res["models"]["random_forest"]["precision_at_50"]
print(f"3) Precision@50 — hand-written rule: {base:.3f}, random forest: {rf:.3f} "
      f"({rf/base:.1f}x more of the top 50 right)")

Installing reportlab...
reportlab installed.
1) Declining rate across all 30,000 pages: 54.2%
2) Visible/addressable pool: 22,006 pages (73.4% of the dataset)
3) Precision@50 — hand-written rule: 0.240, random forest: 0.740 (3.1x more of the top 50 right)


## 4. Careful words: what I can and can't claim

**What I can claim:**
- Observed, correlational patterns in this pseudonymized, aggregated dataset — e.g. a
  hand-written rule versus a learned model differ measurably in Precision@50, on a held-out,
  client-separated split.
- Decision-support: a ranked queue with reason codes that helps an editor triage faster.
- Directional signal about which features associate with decline, within this dataset.

**What I can never claim:**
- That I've reverse-engineered or predicted Google's ranking algorithm — this data only touches
  a client's own performance metrics, never Google's internals.
- Causal proof that refreshing a flagged page fixes it — the model shows association with past
  declines, not that refresh caused past recoveries. There's no experiment here (no A/B test),
  so no causal claim is honest.
- That the label is neutral ground truth — `is_declining_label`/`trend_direction` are themselves
  derived from `trend_pct`, so I have to watch for that when picking features (never using
  `trend_pct` or `trend_direction` as an input, only as the target).

In [6]:
print("Validation split used in my pipeline run:", res["split_strategy"])
print("(pages from a client are never in both train and test — this is what keeps the")
print(" Precision@50 numbers above honest rather than optimistic.)")



Validation split used in my pipeline run: client_holdout
(pages from a client are never in both train and test — this is what keeps the
 Precision@50 numbers above honest rather than optimistic.)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.